In [1]:
import numpy as np
import sympy as sp
from sklearn.base import BaseEstimator, TransformerMixin

# ── Symbolic derivative cache ──────────────────────────────────────────────────

_deriv_cache = {}

def _get_derivatives(N: int, kappa: float):
    """
    Symbolically compute and cache the first N derivatives of
        gamma_tilde(s) = (kappa^2 + s^2)^{-1}
    returning a list of N numpy-callable functions.
    """
    key = (N, kappa)
    if key in _deriv_cache:
        return _deriv_cache[key]

    s = sp.Symbol('s', real=True)
    expr = 1 / (sp.Rational(kappa).limit_denominator(1000)**2 + s**2)
    fns = []
    for _ in range(N):
        fns.append(sp.lambdify(s, sp.simplify(expr), modules='numpy'))
        expr = sp.diff(expr, s)

    _deriv_cache[key] = fns
    return fns


# ── Core helpers ───────────────────────────────────────────────────────────────

def _mat_sqrt_inv(X: np.ndarray) -> np.ndarray:
    """Compute X^{-1/2} via eigendecomposition."""
    L, V = np.linalg.eigh(X)
    return V * (1.0 / np.sqrt(np.clip(L, 1e-12, None)))[None, :] @ V.T


def _cauchy_kernel_single(X: np.ndarray, Y: np.ndarray, kappa: float) -> float:
    """
    Evaluate K(X, Y) = f(X^{-1/2} Y X^{-1/2}) per equation (25).

    f(x) = |det(x)^{(N-1)/2} * det(M) / V(rho)|

    where:
      rho    = sorted eigenvalues of x = X^{-1/2} Y X^{-1/2}
      V(rho) = prod_{k < l} (rho_l - rho_k)   [positive Vandermonde]
      M[k,l] = -gamma_tilde^{(k-1)}(log rho_l)
    """
    x = _mat_sqrt_inv(X) @ Y @ _mat_sqrt_inv(X)
    N = x.shape[0]

    rho = np.sort(np.clip(np.linalg.eigvalsh(x), 1e-12, None))
    log_rho = np.log(rho)

    # Vandermonde: prod_{l > k} (rho_l - rho_k)
    V = np.prod([(rho[l] - rho[k])
                 for k in range(N)
                 for l in range(k + 1, N)])

    # Handle near-degenerate eigenvalues with a small perturbation
    if abs(V) < 1e-10:
        rho = rho + np.linspace(-1e-6, 1e-6, N)
        log_rho = np.log(np.clip(rho, 1e-12, None))
        V = np.prod([(rho[l] - rho[k])
                     for k in range(N)
                     for l in range(k + 1, N)])

    derivs = _get_derivatives(N, kappa)
    M = np.array([[-float(derivs[k](log_rho[l])) for l in range(N)]
                  for k in range(N)])

    det_power = np.prod(rho) ** ((N - 1) / 2.0)
    return abs(float(det_power * np.linalg.det(M) / V))


# ── Scikit-learn transformer ───────────────────────────────────────────────────

class CauchyGramMatrix(BaseEstimator, TransformerMixin):
    """
    Scikit-learn transformer that builds the Gram matrix for the strictly
    positive-definite Cauchy kernel on the SPD manifold (equation 25).

        K(X, Y) = f(X^{-1/2} Y X^{-1/2})

    where f is derived via the Helgason-Fourier (spherical) transform with
    spectral density gamma(t) = (kappa/2) * exp(-kappa|t|).

    This kernel is guaranteed PD by Godement's theorem, unlike the naive
    geodesic substitution k(X,Y) = (kappa^2 + delta^2)^{-l} which fails
    conditional negative definiteness for matrix dimension N >= 2.

    Parameters
    ----------
    kappa : float, default=1.0
        Scale parameter. Must be > 0. Larger kappa = broader kernel.

    Usage (mirrors SteinGramMatrix)
    --------------------------------
    pipe = make_pipeline(
        Covariances(),
        MicrovoltScaler(),
        CauchyGramMatrix(kappa=1.0),
        SVC(kernel='precomputed'),
    )
    """

    def __init__(self, kappa: float = 1.0):
        if kappa <= 0:
            raise ValueError(f"kappa must be > 0, got {kappa}")
        self.kappa = kappa
        self.X_train_ = None

    def fit(self, X: np.ndarray, y=None):
        """
        Store training covariance matrices.

        Parameters
        ----------
        X : ndarray of shape (n_trials, n_channels, n_channels)
            Batch of SPD covariance matrices.
        """
        self.X_train_ = X
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        """
        Compute the (N, M) kernel matrix between X and the stored training set.

        Parameters
        ----------
        X : ndarray of shape (n_trials, n_channels, n_channels)

        Returns
        -------
        K : ndarray of shape (n_trials, n_train_trials)
        """
        N = len(X)
        M = len(self.X_train_)
        K = np.zeros((N, M))

        for i in range(N):
            for j in range(M):
                K[i, j] = _cauchy_kernel_single(X[i], self.X_train_[j], self.kappa)

        return K

In [2]:
import numpy as np
from tqdm.auto import tqdm
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, ShuffleSplit
from pyriemann.estimation import Covariances
from moabb.datasets import BNCI2014001
from moabb.paradigms import MotorImagery
import mne
import sympy as sp

# ── Butterworth paradigm ───────────────────────────────────────────────────────

class ButterworthMotorImagery(MotorImagery):
    def preprocess_raw(self, raw, dataset, fitting_config=None):
        iir_params = dict(order=5, ftype='butter')
        raw.filter(l_freq=self.fmin, h_freq=self.fmax,
                   method='iir', iir_params=iir_params, verbose=False)
        return super().preprocess_raw(raw, dataset, fitting_config)

# ── MicrovoltScaler ────────────────────────────────────────────────────────────

class MicrovoltScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X * 1e6

# ── Stein kernel ───────────────────────────────────────────────────────────────

class SteinGramMatrix(BaseEstimator, TransformerMixin):
    def __init__(self, normalized=False, beta=0.5):
        self.normalized = normalized
        self.beta = beta
        self.X_train_ = None

    def fit(self, X, y=None):
        self.X_train_ = X
        return self

    def transform(self, X):
        N, M = len(X), len(self.X_train_)
        K = np.zeros((N, M))
        if self.normalized:
            log_det_X     = np.array([np.linalg.slogdet(x)[1] for x in X])
            log_det_train = np.array([np.linalg.slogdet(t)[1] for t in self.X_train_])
        for i in range(N):
            for j in range(M):
                M_ij = (X[i] + self.X_train_[j]) / 2.0
                _, log_det_M = np.linalg.slogdet(M_ij)
                if self.normalized:
                    log_K_ij = (-self.beta * log_det_M
                                + (self.beta / 2.0) * log_det_X[i]
                                + (self.beta / 2.0) * log_det_train[j])
                else:
                    log_K_ij = -self.beta * log_det_M
                K[i, j] = np.exp(log_K_ij)
        return K


/Users/rishabhkumar/miniconda3/envs/obsidian/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset  = BNCI2014001()
paradigm = ButterworthMotorImagery(fmin=8.0, fmax=30.0, tmin=0.5, tmax=2.5)

cv_strategy = ShuffleSplit(n_splits=30, test_size=0.2, random_state=42)

# Separate param grids — C for SVC, kappa for Cauchy
param_grid_svc    = {'svc__C': [0.1, 1, 10, 100, 1000]}
param_grid_cauchy = {'svc__C': [0.1, 1, 10, 100, 1000],
                     'cauchygrammatrix__kappa': [0.1, 0.5, 1.0, 2.0, 5.0]}

pipeline_std    = make_pipeline(Covariances(estimator='scm'),
                                SteinGramMatrix(normalized=False),
                                SVC(kernel='precomputed'))

pipeline_norm   = make_pipeline(Covariances(estimator='scm'),
                                SteinGramMatrix(normalized=True),
                                SVC(kernel='precomputed'))

pipeline_cauchy = make_pipeline(Covariances(estimator='scm'),
                                CauchyGramMatrix(kappa=1.0),
                                SVC(kernel='precomputed'))

grid_std    = GridSearchCV(pipeline_std,    param_grid_svc,    cv=cv_strategy, n_jobs=-1)
grid_norm   = GridSearchCV(pipeline_norm,   param_grid_svc,    cv=cv_strategy, n_jobs=-1)
grid_cauchy = GridSearchCV(pipeline_cauchy, param_grid_cauchy, cv=cv_strategy, n_jobs=-1)

# ── Evaluation loop ────────────────────────────────────────────────────────────

results_std, results_norm, results_cauchy = [], [], []
subjects = [1, 2, 3, 4, 5, 6, 7, 8, 9]

for subject in tqdm(subjects, desc='Processing subjects'):
    X, y, metadata = paradigm.get_data(dataset, subjects=[subject])

    train_idx = metadata['session'] == '0train'
    test_idx  = metadata['session'] == '1test'
    X_train, y_train = X[train_idx], y[train_idx]
    X_test,  y_test  = X[test_idx],  y[test_idx]

    grid_std.fit(X_train, y_train)
    results_std.append(grid_std.score(X_test, y_test))

    grid_norm.fit(X_train, y_train)
    results_norm.append(grid_norm.score(X_test, y_test))

    grid_cauchy.fit(X_train, y_train)
    results_cauchy.append(grid_cauchy.score(X_test, y_test))

    print(f"Sub {subject:02d}  "
          f"Stein: {results_std[-1]*100:5.2f}%  "
          f"Stein-Norm: {results_norm[-1]*100:5.2f}%  "
          f"Cauchy: {results_cauchy[-1]*100:5.2f}%  "
          f"[best kappa={grid_cauchy.best_params_['cauchygrammatrix__kappa']}  "
          f"C={grid_cauchy.best_params_['svc__C']}]")

print(f"\n{'─'*60}")
print(f"Mean  Stein:      {np.mean(results_std)*100:.2f}%  ± {np.std(results_std)*100:.2f}%")
print(f"Mean  Stein-Norm: {np.mean(results_norm)*100:.2f}%  ± {np.std(results_norm)*100:.2f}%")
print(f"Mean  Cauchy:     {np.mean(results_cauchy)*100:.2f}%  ± {np.std(results_cauchy)*100:.2f}%")

In [8]:
!pip install --upgrade numpy scikit-learn scipy

In [7]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
